# ELECTRO-VOLT — 03. Modelado y Comparativa de Algoritmos ML
**Proyecto Integrador · Maestría en IA · UEES (2026)**
**Autores:** Jordan Vera Coello & Jorge Estrella Frere

En este notebook se entrena y evalúa el catálogo de modelos supervisados:
1. **Regresión Logística con StandardScaler:** Modelo lineal baseline.
2. **Random Forest (100 estimadores):** Modelo ensamble principal.
3. **SVM con Kernel RBF y StandardScaler:** Modelo no lineal de referencia.
4. Comparativa exhaustiva (Accuracy Train, Accuracy Test, Brecha/Gap, F1-macro y Tiempo).
5. Evaluación contra la Meta SMART ($F1 \ge 0.90$).
6. Validación cruzada estratificada `StratifiedKFold (k=5)`.


In [ ]:
import sys
from pathlib import Path

raiz = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(raiz) not in sys.path:
    sys.path.append(str(raiz))

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from src.data_processing import cargar_datos, mapear_clases, dividir_datos
from src.model import crear_modelos
from src.train import entrenar_y_comparar_todos, ejecutar_validacion_cruzada
from src.evaluate import graficar_comparativa_f1
from src.utils import COLUMNAS, SEMILLA

sns.set_theme(style="whitegrid")
print("[OK] Módulos de modelado importados.")


## 1. Carga y partición de datos (5 Clases)


In [ ]:
df = mapear_clases(cargar_datos(), modo="5_clases")
X_ent, X_pru, y_ent, y_pru, idx_ent, idx_pru = dividir_datos(df, test_size=0.20, random_state=SEMILLA)
print(f"Train: {X_ent.shape} | Test: {X_pru.shape}")


## 2. Entrenamiento y comparativa de los 3 Modelos
Entrenamos los modelos cronometrando tiempos y calculando brechas de generalización.


In [ ]:
tabla_comparativa, modelos_entrenados = entrenar_y_comparar_todos(
    X_ent, y_ent, X_pru, y_pru, random_state=SEMILLA
)
print("=== TABLA COMPARATIVA DE MODELOS ===")
tabla_comparativa.round(4)


## 3. Visualización: F1-Macro vs. Meta SMART (0.90)
Graficamos el rendimiento de los modelos frente a la línea de umbral exigida.


In [ ]:
fig = graficar_comparativa_f1(tabla_comparativa)
plt.show()


## 4. Validación cruzada estratificada (StratifiedKFold, k = 5)
Verificamos que el rendimiento sea estable en diferentes particiones aleatorias con `shuffle=True`.


In [ ]:
modelos_cv = {
    "Random Forest": crear_modelos(random_state=SEMILLA)["Random Forest"],
    "SVM (RBF)": crear_modelos(random_state=SEMILLA)["SVM (RBF)"],
}
res_cv = ejecutar_validacion_cruzada(modelos_cv, df[COLUMNAS], df["clase"], n_splits=5)

for m_nom, d in res_cv.items():
    print(f"{m_nom}: F1-macro = {d['media']:.4f} ± {d['std']:.4f}  (Pliegues: {[round(x, 4) for x in d['pliegues']]})")
